In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import r2_score, mean_squared_error
import shap

# ---------- 0. 文件路径 ----------
DATA_FILE      = "safety_scored.xlsx"
UNSCORED_FILE  = "images.xlsx"
OUT_DIR        = Path("D:\Desktop\output_safety")
OUT_DIR.mkdir(exist_ok=True)

PREDICT_XLSX   = OUT_DIR / "images_predicted_safety.xlsx"
BSWARM_PNG     = OUT_DIR / "shap_beeswarm_safety.png"
BAR_PNG        = OUT_DIR / "shap_bar_safety.png"

TARGET_COL   = "Safety"
FEATURE_COLS = ["Road","Building","Pole Group","Indicator","Vegetation","Sky",
                "Person","Car","Motorcycle","Bicycle","Clothes","Trash Can",
                "Riverway","Signboard","Air Conditioner Condenser","Festival Elements"]

# ---------- 1. 读取数据 ----------
df = pd.read_excel(DATA_FILE)
df[TARGET_COL] = df[TARGET_COL].round(5)

X = df[FEATURE_COLS].copy()
rng = np.random.default_rng(42)
for col in FEATURE_COLS:
    mask0 = X[col] == 0
    if mask0.any():
        X.loc[mask0, col] += rng.uniform(0.01, 0.03, size=mask0.sum())

y = df[TARGET_COL].values

# ---------- 2. 划分 & 训练随机森林 ----------
X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(
        n_estimators     = 400,
        max_depth        = 9,
        min_samples_leaf = 3,        
        max_features     = "sqrt",
        random_state     = 42
     )
rf.fit(X_tr, y_tr)

# ---------- 3. 单调校准  ----------
iso = IsotonicRegression(
        y_min=y.min(),               
        y_max=y.max(),
        increasing=True,
        out_of_bounds="clip"         
     )
iso.fit(rf.predict(X), y)

# ---------- 4. 评估（使用校准后分数） ----------
def _met(t, p): return r2_score(t, p), mean_squared_error(t, p, squared=False)

raw_tr, raw_te = rf.predict(X_tr), rf.predict(X_te)
cal_tr, cal_te = iso.transform(raw_tr), iso.transform(raw_te)

r2_tr,  rmse_tr  = _met(y_tr,  cal_tr)
r2_te,  rmse_te  = _met(y_te,  cal_te)
r2_all, rmse_all = _met(y,     iso.transform(rf.predict(X)))


print(f"Train   R² = {r2_tr :.4f} | RMSE = {rmse_tr :.4f}")
print(f"Test    R² = {r2_te :.4f} | RMSE = {rmse_te :.4f}")
print(f"Overall R² = {r2_all:.4f} | RMSE = {rmse_all:.4f}")


# ---------- 5. SHAP ----------
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X)

plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(shap_values, X, feature_names=FEATURE_COLS, show=False)
plt.title("SHAP Summary (Beeswarm) – Safety")
plt.savefig(BSWARM_PNG, bbox_inches="tight", dpi=300)
plt.close()

plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(shap_values, X, feature_names=FEATURE_COLS,
                  plot_type="bar", show=False)
plt.title("Feature Importance (mean |SHAP value|)")
plt.savefig(BAR_PNG, bbox_inches="tight", dpi=300)
plt.close()

# ---------- 6. 预测未打分样本 ----------
df_new = pd.read_excel(UNSCORED_FILE)
X_new  = df_new[FEATURE_COLS].copy()
for col in FEATURE_COLS:
    m0 = X_new[col] == 0
    if m0.any():
        X_new.loc[m0, col] += rng.uniform(0.01, 0.03, size=m0.sum())

raw_pred   = rf.predict(X_new)
df_new[TARGET_COL] = iso.transform(raw_pred).round(5)  
df_new.to_excel(PREDICT_XLSX, index=False, float_format="%.5f")

print("✔ 预测文件:", PREDICT_XLSX)
print("✔ SHAP 图:", BSWARM_PNG, BAR_PNG)



=== RF + Isotonic (Safety) ===
Train   R² = 0.9733 | RMSE = 0.0871
Test    R² = 0.9011 | RMSE = 0.1682
Overall R² = 0.9588 | RMSE = 0.1083

✔ 预测文件: D:\Desktop\output_safety\images_predicted_safety.xlsx
✔ SHAP 图: D:\Desktop\output_safety\shap_beeswarm_safety.png D:\Desktop\output_safety\shap_bar_safety.png
